<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day20_practice1_LLM_API_%ED%94%84%EB%A1%AC%ED%94%84%ED%8A%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# LLM API - 만들어진 거인을 불러 쓰는 법
# 프롬프트 엔지니어링 4원칙: 역할·지시·예시·형식

In [13]:
import requests, os, json, time, random

In [21]:
# 셀 1. API 호출 - openAI 호환 api 형식으로 코드 작성

BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
MODELS = ["gemini-flash-lite-latest"] # Google이 업데이트하는 Flash-Lite 라인업의 가장 최신 버전으로 자동 연결

In [15]:
# API 키 - Colab Secrets에서 가져오기
try:
  from google.colab import userdata
  API_KEY = userdata.get("VISION_API_KEY")
except Exception:
  API_KEY = os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

In [16]:
RETRY_STATUS = {429, 500, 502, 503, 504} # 잠깐 뒤 다시 시도하면 되는 상태 코드, 429 요청과다, 500번대는 서버문제
MAX_RETRIES = 4 # 모델당 최대 시도 횟수

# 어떤 대화든 넣으면 답만 돌려주는 범용 호출 함수
def call_llm(messages, temperature=0.2): # 0.0~0.3 : 사실 추출, 분류, JSON / 0.7~1.0 : 창작, 브레인스토밍
  if not API_KEY:
    return None
  for model in MODELS:
    for attempt in range(MAX_RETRIES):
      try:
        r = requests.post(
            f"{BASE_URL}/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}"},
            json={"model": model, "messages": messages, "temperature": temperature},
            timeout=60,
        )
        try:
          print("본문:")
          print(json.dumps(r.json(), ensure_ascii=False, indent=2)) # 응답 본문 JSON 문자열
        except ValueError:
          print("본문(JSON 아님):", r.text[:500]) # 원문 그대로 출력
        if r.status_code not in RETRY_STATUS:
          r.raise_for_status() # 즉시 예외
          return r.json()["choices"][0]["message"]["content"] # 서버가 보낸 응답 본문(JSON 형식의 글자)을 파이썬 dict로
        print(f"{model} {r.status_code} (시도 {attempt+1}/{MAX_RETRIES})")
      except (requests.Timeout, requests.ConnectionError) as e:
        print(f"{model} 네트워크 오류: {type(e).__name__}(시도 {attempt+1}/{MAX_RETRIES})")
      except Exception as e: # 인증 오류를 재시도해도 소용없는 실패
        print("API 실패:", e)
        return None
      if attempt < MAX_RETRIES - 1:
        time.sleep(2 ** attempt + random.uniform(0, 0.5)) # 대시시간을 조금씩 늘려감
    print(f"{model} 계속 실패 → 다음 모델 시도")
  print("모든 모델 실패 → 모의 응답으로 대체")
  return None

In [20]:
# 셀 2. 프롬프트 4원칙 - 나쁜 예 vs 좋은 예
신고문 = "아니 어제부터 강남대로 사거리쪽에 큰 구멍이 있어서 차가 지나갈때마다 쾅쾅거리는데 이거 진짜 사고나겠어요 빨리좀 어떻게 해주세요"
bad_prompt = [
    {"role": "user",
     "content": f"이거 분석해줘: {신고문}"}
]
good_prompt = [
    {"role": "system",
     "content": "너는 도로 위험 신고 접수 시스템의 분석 담당이다." # 역할
     "감정적 표현은 무시하고 사실만 추출한다."},
    {"role": "user",
     "content": f"""다음 신고 문장을 분석하라.
     [예시]
     신고: '맨홀 뚜껑이 반쯤 열려있어요. 위험해요'
     분석: 유형=맨홀, 위치=미상, 심각도=3, 요약=맨홀 뚜껑 반개방
     [분석할 신고]
     {신고문}
     [출력 형식] 유형, 위치, 심각도(1~5), 한 줄 요약만, 사족 금지
     """ # 지시
     # few-shot 예시
     }
]
print("[프롬프트 4원칙] 1.역할(system) 2.지시 3.예시(few-shot) 4.형식-좋은 프롬프트의 뼈대")
answer = call_llm(good_prompt)
if answer is None:
  answer = "유형=포트홀, 위치=강남대로 사거리, 심각도=4, 요약=사거리 대형 포트홀로 차량 충격 발생_미리 준비한 가짜 답"
print("응답:", answer)

[프롬프트 4원칙] 1.역할(system) 2.지시 3.예시(few-shot) 4.형식-좋은 프롬프트의 뼈대
본문:
[
  {
    "error": {
      "code": 503,
      "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
      "status": "UNAVAILABLE"
    }
  }
]
gemini-flash-latest 503 (시도 1/4)
본문:
[
  {
    "error": {
      "code": 503,
      "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
      "status": "UNAVAILABLE"
    }
  }
]
gemini-flash-latest 503 (시도 2/4)
gemini-flash-latest 네트워크 오류: ReadTimeout(시도 3/4)
본문:
[
  {
    "error": {
      "code": 503,
      "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
      "status": "UNAVAILABLE"
    }
  }
]
gemini-flash-latest 503 (시도 4/4)
gemini-flash-latest 계속 실패 → 다음 모델 시도
본문:
{
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {